Área ardida de 2025 por classe

In [ ]:
from pathlib import Path
import sys
import pandas as pd

sys.path.append("/code/scripts")

from analysis_config import ANALYSIS
from burned_area_analysis_utils import (
    burned_area_by_class,
    burned_area_top_classes,
    compare_binary_rasters
)
from pilot_utils import clip_raster

In [ ]:
references = {
    "fundao": {
        "ICNF_2025": (
            "/code/data/processed/centro/area_ardida/icnf/"
            "raster_binary/rst_ba_2025_bin.tif"
        ),
        "EFFIS_2025": (
            "/code/data/processed/centro/area_ardida/effis/"
            "raster_binary/rst_ba_2025_bin.tif"
        ),
        "EFFIS_summer_2025": (
            "/code/data/processed/centro/area_ardida/effis/"
            "raster_target_yearly/rst_ba_2025_target.tif"
        )
    },
    "badajoz": {
        "EFFIS_2025": (
            "/code/data/processed/extremadura/area_ardida/effis/"
            "raster_binary/rst_ba_2025_bin.tif"
        ),
        "EFFIS_summer_2025": (
            "/code/data/processed/extremadura/area_ardida/effis/"
            "raster_target_yearly/rst_ba_2025_target.tif"
        )
    }
}

In [ ]:
all_support = []
all_distribution = []
reference_checks = []

for pilot, pilot_references in references.items():
    pilot_dir = ANALYSIS / "02_classes" / pilot
    inventory = pd.read_csv(
        pilot_dir / f"class_inventory_{pilot}.csv"
    )

    municipality = Path(inventory.iloc[0]["municipality"])
    out_dir = ANALYSIS / "08_burned_area_2025" / pilot
    reference_dir = out_dir / "references"
    reference_dir.mkdir(parents=True, exist_ok=True)

    local_references = {}

    for reference_name, regional_reference in pilot_references.items():
        output_reference = reference_dir / f"{reference_name}_{pilot}.tif"
        clip_raster(
            regional_reference,
            municipality,
            output_reference
        )
        local_references[reference_name] = str(output_reference)

    reference_checks.append(
        compare_binary_rasters(
            local_references["EFFIS_2025"],
            local_references["EFFIS_summer_2025"],
            pilot=pilot,
            reference_a="EFFIS_2025",
            reference_b="EFFIS_summer_2025"
        )
    )

    structural = inventory.loc[
        inventory["product"].isin(["susceptibility", "hazard"])
    ]

    for _, record in structural.iterrows():
        reference_names = (
            ["ICNF_2025", "EFFIS_2025"]
            if pilot == "fundao"
            else ["EFFIS_2025"]
        )

        for reference_name in reference_names:
            support, distribution = burned_area_by_class(
                record["local_class_raster"],
                local_references[reference_name],
                pilot=pilot,
                map_id=record["map_id"],
                scenario=record["scenario"],
                product=record["product"],
                reference=reference_name
            )
            all_support.append(support)
            all_distribution.append(distribution)

    seasonal = inventory.loc[
        inventory["product"].isin([
            "seasonal_susc",
            "ssr",
            "seasonal_combined"
        ])
    ]

    for _, record in seasonal.iterrows():
        support, distribution = burned_area_by_class(
            record["local_class_raster"],
            local_references["EFFIS_summer_2025"],
            pilot=pilot,
            map_id=record["map_id"],
            scenario=record["scenario"],
            product=record["product"],
            reference="EFFIS_summer_2025"
        )
        all_support.append(support)
        all_distribution.append(distribution)

In [ ]:
support = pd.concat(all_support, ignore_index=True)
distribution = pd.concat(all_distribution, ignore_index=True)
top_classes = burned_area_top_classes(distribution)
reference_check = pd.concat(reference_checks, ignore_index=True)

out_dir = ANALYSIS / "08_burned_area_2025"
out_dir.mkdir(parents=True, exist_ok=True)
output = out_dir / "burned_area_2025_by_class.xlsx"

with pd.ExcelWriter(output) as writer:
    support.to_excel(
        writer,
        sheet_name="Local_support",
        index=False
    )
    distribution.to_excel(
        writer,
        sheet_name="Burned_by_class",
        index=False
    )
    top_classes.to_excel(
        writer,
        sheet_name="Top_classes_summary",
        index=False
    )
    reference_check.to_excel(
        writer,
        sheet_name="Reference_check",
        index=False
    )

print("Resultados guardados em:", output)
reference_check